<a href="https://colab.research.google.com/github/GillValenzuela/curso_data_science/blob/master/DS_Ingemat_Clase_22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install torch torchvision grad-cam pillow requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 57.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 207.5/207.5 MB 185.7 MB/s eta 0:00:01

In [ ]:
!pip install -q --upgrade grad-cam

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import matplotlib.pyplot as plt, torch, requests, io
from torchvision import transforms, utils, datasets
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm.auto import tqdm

In [ ]:
# --- Data ---
tfms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))
])
train_ds = datasets.CIFAR10("data", train=True, download=True, transform=tfms)
test_ds  = datasets.CIFAR10("data", train=False, transform=tfms)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
test_dl  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=0)

In [ ]:
device   = "cuda" if torch.cuda.is_available() else "cpu"
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,1,1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,1,1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,1,1),nn.BatchNorm2d(128),nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 10)
        )
    def forward(self,x): return self.classifier(self.features(x))

model = SimpleCNN().to(device)
print(f"Parámetros: {sum(p.numel() for p in model.parameters()):,}")

In [7]:
# -----------------------------------------------------------
# 4. Pérdida y optimizador
# -----------------------------------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# -----------------------------------------------------------
# 5. Entrenamiento breve (5 épocas) + validación
# -----------------------------------------------------------
def accuracy(net, loader):
    net.eval(); correct = total = 0
    with torch.no_grad():
        for xb,yb in loader:
            pred = net(xb.to(device)).argmax(1).cpu()
            correct += (pred == yb).sum().item()
            total   += yb.size(0)
    return correct/total

epochs = 5
for ep in range(1, epochs+1):
    model.train()
    for xb,yb in tqdm(train_dl, desc=f"Epoch {ep}/{epochs}", leave=False):
        xb,yb = xb.to(device), yb.to(device)
        loss  = criterion(model(xb), yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    val_acc = accuracy(model, test_dl)
    print(f"Época {ep:02d}  |  val acc: {val_acc*100:.2f}%")

# -----------------------------------------------------------
# 6. Guardar el modelo
# -----------------------------------------------------------
torch.save(model, "cnn_cifar10.pt")
print("✅ Modelo guardado como cnn_cifar10.pt en el directorio de Colab")

Epoch 1/5:   0%|          | 0/1563 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# 1) Cargar la CNN casera entrenada anteriormente
model = torch.load("cnn_cifar10.pt", map_location="cpu",weights_only=False).eval()

# 2) Tomar un lote de prueba (del DataLoader que ya tenemos)
imgs, labels = next(iter(test_dl))
img, label = imgs[1:2], labels[1]

# 3) Preparar Grad-CAM sobre la última conv de nuestra red
target_layer = model.features[-3]          # última Conv (64/128 filtros)
img = img.to(device)

with GradCAM(model=model,
             target_layers=[target_layer]) as cam:
    grayscale_cam = cam(input_tensor=img)[0]

# 4) Des-normalizar para visualizar
def denorm(t):
    mean = torch.tensor((0.4914,0.4822,0.4465)).view(3,1,1)
    std  = torch.tensor((0.2023,0.1994,0.2010)).view(3,1,1)
    return torch.clamp(t * std + mean, 0, 1)

rgb = denorm(img.cpu()[0]).permute(1,2,0).numpy()
cam_img = show_cam_on_image(rgb, grayscale_cam, use_rgb=True)

classes = test_ds.classes           # ['airplane', 'automobile', …, 'truck']
model.to(device).eval()
pred_idx = model(img).argmax(1).item()
true_idx = label.item()
pred = model(img).argmax(1).item()


true = label.item()
title = f"Pred: {classes[pred]}   |   True: {classes[true]}"

plt.figure(figsize=(6,3))
plt.subplot(1,2,1); plt.imshow(rgb);     plt.axis("off"); plt.title("Original")
plt.subplot(1,2,2); plt.imshow(cam_img); plt.axis("off"); plt.title("Grad-CAM")
plt.suptitle(title, fontsize=12)
plt.show()